# Download Sept-Nov 2025 Harbin Sentinel-2 scenes (Colab)

Runs `download-haerbing-sepnov-sentinel2-data.py` from Colab instead of your local machine. The script itself lives in the repo and is unchanged -- this notebook just handles two things Colab needs that your local `venv` run didn't:

1. **Credentials**: `.env.cdse` is deliberately gitignored (a CDSE username/password should never be pushed to a public repo -- see the file's own setup comment), so it isn't in the cloned repo on Colab. Fix: upload your local `.env.cdse` to Google Drive once, next to your dataset tars, and this notebook loads it from there into the session's environment variables -- it is never written to the Colab VM's disk or printed anywhere in this notebook's output.
2. **Output location**: the script downloads scenes to `Haerbing_Dataset_sepnov/` on local disk. Colab's disk is wiped when the runtime disconnects, so this notebook copies the finished result to Drive at the end -- do this in the *same* runtime as the download (not a fresh session later).

**One-time setup before running this notebook**: in Google Drive, go to the same `COMP0173/` folder you uploaded the ground-truth tars to, and upload your local `.env.cdse` file there (Drive web UI: drag and drop, or right-click > File upload). It's a two-line text file, no need to tar it.

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = "/content/COMP0173_poster_pre"
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/jy-gfm/COMP0173_poster_pre.git {REPO_DIR}
    os.chdir(REPO_DIR)

    DRIVE_DIR = "/content/drive/MyDrive/COMP0173/"
else:
    DRIVE_DIR = "./"


## Load credentials from Drive (not from the repo -- `.env.cdse` is gitignored on purpose)

In [ ]:
def load_env_file(path):
    if not os.path.exists(path):
        raise RuntimeError(
            f"{path} not found. Upload your local .env.cdse to the COMP0173/ "
            "folder in Google Drive first (see this notebook's opening cell)."
        )
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

load_env_file(os.path.join(DRIVE_DIR, ".env.cdse"))
assert os.environ.get("CDSE_USERNAME") and os.environ.get("CDSE_PASSWORD"), "credentials didn\'t load"
print("Credentials loaded into this session only -- not written to disk, not printed.")


## Run the download script

Same script as the local `venv` version -- `download-haerbing-sepnov-sentinel2-data.py` already reads `CDSE_USERNAME`/`CDSE_PASSWORD` from the environment, which the cell above just populated. This can take a while (14+ scenes, ~1GB each) -- if the runtime disconnects partway through, just re-run this cell: the script skips any `.SAFE` folder it already downloaded.

In [ ]:
!python download-haerbing-sepnov-sentinel2-data.py


## Copy the result to Drive

Do this before the runtime disconnects -- `Haerbing_Dataset_sepnov/` on Colab's local disk does not survive a disconnect. A single tar archive is more reliable to upload/download than thousands of individual files (same reasoning as every other dataset in this project).

In [ ]:
if IN_COLAB:
    TAR_NAME = "Haerbing_Dataset_sepnov"
    !tar -cf {TAR_NAME}.tar {TAR_NAME}
    import shutil
    shutil.copy(f"{TAR_NAME}.tar", DRIVE_DIR)
    print(f"Saved {TAR_NAME}.tar to {DRIVE_DIR}")
